# Send Data To Qdrant

This example reads the local sample text file, chunks it, creates sentence-transformer embeddings, and upserts the points into Qdrant.

Set `QDRANT_URL` and `QDRANT_API_KEY` in your environment to send data to a hosted Qdrant instance. If those variables are missing, the code uses a local on-disk Qdrant database instead.

In [20]:
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue, PayloadSchemaType
from dotenv import load_dotenv
load_dotenv()
import os

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

if QDRANT_URL and QDRANT_API_KEY:
    client = QdrantClient(
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
    )
else:
    local_qdrant_path = Path.cwd() / "qdrant_local"
    client = QdrantClient(path=str(local_qdrant_path))

In [30]:
COLLECTION_NAME = "company_report_chunks"

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="company_name",
    field_schema=PayloadSchemaType.KEYWORD,
    wait=True,
 )

client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="year",
    field_schema=PayloadSchemaType.INTEGER,
    wait=True,
 )

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

In [28]:
from uuid import uuid4
from sentence_transformers import SentenceTransformer

DATA_DIR = Path.cwd()
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MAX_WORDS_PER_CHUNK = 80
CHUNK_OVERLAP_WORDS = 15

In [29]:
def parse_document(file_path: Path) -> dict:
    raw_text = file_path.read_text(encoding="utf-8").strip()
    header_text, body_text = raw_text.split("\n\n", 1)

    metadata = {}
    for line in header_text.splitlines():
        key, value = line.split(":", 1)
        metadata[key.strip()] = value.strip()

    return {
        "company_name": metadata["company_name"],
        "year": int(metadata["year"]),
        "body": body_text.strip(),
    }


def chunk_text(
    text: str,
    max_words_per_chunk: int = MAX_WORDS_PER_CHUNK,
    overlap_words: int = CHUNK_OVERLAP_WORDS,
 ) -> list[str]:
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + max_words_per_chunk, len(words))
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)

        if end >= len(words):
            break

        start = max(end - overlap_words, start + 1)

    return chunks


def build_chunk_records(data_dir: Path) -> list[dict]:
    chunk_records = []

    for file_path in sorted(data_dir.glob("*.txt")):
        document = parse_document(file_path)
        chunks = chunk_text(document["body"])

        for chunk_number, chunk in enumerate(chunks, start=1):
            chunk_records.append(
                {
                    "id": str(uuid4()),
                    "text": chunk,
                    "company_name": document["company_name"],
                    "year": document["year"],
                    "chunk_number": chunk_number,
                }
            )

    return chunk_records


chunk_records = build_chunk_records(DATA_DIR)
len(chunk_records), chunk_records[0]

(11,
 {'id': 'a46d6cde-f1ae-4cbe-b925-da163f521652',
  'text': "Amazon is a multinational technology company best known for e-commerce, Amazon Web Services (AWS), digital advertising, logistics, devices, and streaming services. The company was founded by Jeff Bezos in 1994, and Andy Jassy served as chief executive officer in 2023. Amazon's operating model combines first-party retail, a large third-party marketplace, subscription services such as Prime, and a fast-growing cloud business through AWS. In 2023, Amazon reported approximately $574.7 billion in revenue and about $30.4 billion in net income. The year",
  'company_name': 'Amazon',
  'year': 2023,
  'chunk_number': 1})

In [24]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4481.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
vectors = embedding_model.encode(
    [record["text"] for record in chunk_records],
    normalize_embeddings=True,
    show_progress_bar=False,
).tolist()

points = [
    PointStruct(
        id=record["id"],
        vector=vector,
        payload={
            "text": record["text"],
            "company_name": record["company_name"],
            "year": record["year"],
        },
    )
    for record, vector in zip(chunk_records, vectors)
]

client.upsert(collection_name=COLLECTION_NAME, points=points)
len(points)

11

In [34]:
import re

available_companies = sorted({record["company_name"] for record in chunk_records})
available_years = sorted({record["year"] for record in chunk_records})
company_aliases = {company.lower(): company for company in available_companies}
company_aliases.update({
    "amazon": "Amazon",
    "aws": "Amazon",
    "microsoft": "Microsoft",
    "msft": "Microsoft",
})


def infer_filters_from_query(query_text: str) -> tuple[str | None, int | None]:
    lowered_query = query_text.lower()
    detected_company = None

    for alias, canonical_name in company_aliases.items():
        if alias in lowered_query:
            detected_company = canonical_name
            break

    detected_year = None
    year_matches = re.findall(r"\b(20\d{2})\b", query_text)
    for year_match in year_matches:
        year_value = int(year_match)
        if year_value in available_years:
            detected_year = year_value
            break

    return detected_company, detected_year


def build_metadata_filter(
    company_name: str | None = None,
    year: int | None = None,
 ) -> Filter | None:
    must_conditions = []

    if company_name:
        must_conditions.append(
            FieldCondition(key="company_name", match=MatchValue(value=company_name))
        )

    if year is not None:
        must_conditions.append(
            FieldCondition(key="year", match=MatchValue(value=year))
        )

    if not must_conditions:
        return None

    return Filter(must=must_conditions)


def search_reports(
    query_text: str,
    company_name: str | None = None,
    year: int | None = None,
    limit: int = 3,
 ) -> dict:
    inferred_company_name, inferred_year = infer_filters_from_query(query_text)
    selected_company_name = company_name or inferred_company_name
    selected_year = year if year is not None else inferred_year
    query_vector = embedding_model.encode(query_text, normalize_embeddings=True).tolist()

    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=build_metadata_filter(selected_company_name, selected_year),
        limit=limit,
        with_payload=True,
    )

    return {
        "query": query_text,
        "selected_company_name": selected_company_name,
        "selected_year": selected_year,
        "results": [
            {
                "score": point.score,
                "company_name": point.payload["company_name"],
                "year": point.payload["year"],
                "text": point.payload["text"],
            }
            for point in response.points
        ],
    }


available_companies, available_years

(['Amazon', 'Microsoft'], [2023, 2024])

In [38]:
query = "What did mfts do in AI and cloud "

search_reports(query_text=query, limit=3)

{'query': 'What did mfts do in AI and cloud ',
 'selected_company_name': None,
 'selected_year': None,
 'results': [{'score': 0.43086225,
   'company_name': 'Microsoft',
   'year': 2023,
   'text': 'AI tooling across the stack. Microsoft also continued to position Azure, data platforms, developer tools, and productivity applications as core assets for enterprise AI adoption.'},
  {'score': 0.36401418,
   'company_name': 'Amazon',
   'year': 2024,
   'text': "in net income. The year showed continued top-line growth and a substantial improvement in earnings. Amazon benefited from stronger operating efficiency, improved retail margins, and ongoing expansion in cloud and advertising. During 2024, Amazon continued investing in AI-related infrastructure, AWS capabilities, and logistics optimization. AWS remained central to Amazon's enterprise position as organizations increased spending on cloud, data platforms, and generative AI workloads. Amazon also maintained a broad ecosystem that inclu